# Track A: Interpolation Smoothness on Synthetic Physics

**Purpose**: Test whether SmallBERT (Transformer) and SmallMamba (SSM) embedding
spaces respond **smoothly** to continuous input interpolation on a physics problem
(damped harmonic oscillator).

## What This Tests

Given two distinct oscillator trajectories **A** and **B**, we linearly interpolate
between them in input space:

$$\text{interp}(\alpha) = \text{round}((1 - \alpha) \cdot A + \alpha \cdot B)$$

for $\alpha \in [0, 0.01, \ldots, 1.0]$ (101 steps).

Each interpolated sequence is embedded, producing a **path through embedding space**.
We then compute the **Lipschitz profile** -- the local rate of change at each step:

$$L_i = \frac{\| f(\text{interp}(\alpha_{i+1})) - f(\text{interp}(\alpha_i)) \|}{\Delta \alpha}$$

- **Smooth profile** = the embedding manifold is well-behaved and continuous
- **Spiky profile** = "phase transitions" / discontinuities in the embedding space

## Setup Instructions

1. Upload `evaluation_harness.py` to `/content/`
2. Change Runtime to **GPU** (Runtime > Change runtime type > GPU)
3. Run cells in order (~25 min total on A100)

---

In [ ]:
# Install Dependencies

print('Installing core dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy

print('\nBuilding mamba-ssm from source (one-time ~10 min compile)...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK (native CUDA)')
except ImportError:
    print('mamba-ssm: FAILED -- will use pure-PyTorch SSM fallback')

import torch
print(f'\ntorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

import os
import sys
sys.path.insert(0, '.')

# --- Output paths ---
OUTPUT_BASE = './results/interpolation_track_a_319/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# --- Dataset parameters ---
SEQ_LEN    = 512
N_BINS     = 256
N_TRAIN    = 50_000
N_EVAL     = 2_000
SEED       = 320

# --- Special tokens ---
MASK_TOKEN = N_BINS
PAD_TOKEN  = N_BINS + 1
VOCAB_SIZE = N_BINS + 2

# --- Architecture parameters (matched at ~2M params) ---
D_MODEL    = 256
N_LAYERS   = 4
N_HEADS    = 4
FFN_DIM    = 1024
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2

# --- Training parameters ---
# Training uses CLM (causal language modeling) for consistency with
# Architectural Control and Coupled Lorenz experiments.
LR         = 3e-4
WEIGHT_DECAY = 0.01
EPOCHS     = 20
BATCH_SIZE = 64

# --- Interpolation parameters ---
N_INTERP_STEPS = 101   # alpha from 0.0 to 1.0 in 0.01 increments
N_INTERP_PAIRS = 10    # number of (A, B) pairs to average over

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ARCHITECTURES = ['SmallBERT', 'SmallMamba', 'SmallStripedHyena']

print(f'Device: {DEVICE}')
print(f'Interpolation: {N_INTERP_STEPS} steps, {N_INTERP_PAIRS} pairs')
print(f'Output: {OUTPUT_BASE}')

In [ ]:
# Damped Oscillator Dataset Generator
# Uses dataset-global min/max for discretization.

import numpy as np


def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    '''Map continuous values to integer bins in [0, n_bins-1].

    Uses dataset-global min/max when provided so the same physical
    state maps to the same bin across all sequences.
    '''
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12:
        return np.full_like(values, n_bins // 2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    bins = np.clip((normed * (n_bins - 1)).astype(np.int64), 0, n_bins - 1)
    return bins


def generate_oscillators(n_sequences, seq_len=SEQ_LEN, seed=SEED,
                         global_range=None):
    '''x(t) = A * exp(-gamma*t) * cos(omega*t + phi) with random params.

    Returns (sequences, global_range).
    '''
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 4, seq_len, endpoint=False)

    raw_signals = []
    for _ in range(n_sequences):
        A     = rng.uniform(0.5, 2.0)
        gamma = rng.uniform(0.2, 2.0)
        omega = rng.uniform(2.0, 20.0)
        phi   = rng.uniform(0, 2 * np.pi)
        signal = A * np.exp(-gamma * t) * np.cos(omega * t + phi)
        raw_signals.append(signal)

    if global_range is None:
        all_vals = np.concatenate(raw_signals)
        global_range = (float(all_vals.min()), float(all_vals.max()))

    sequences = [discretize(s, global_min=global_range[0], global_max=global_range[1])
                 for s in raw_signals]
    return np.array(sequences, dtype=np.int64), global_range


def generate_oscillator_continuous(A, gamma, omega, phi, seq_len=SEQ_LEN):
    '''Generate a single continuous (un-discretized) oscillator for plotting.'''
    t = np.linspace(0, 4, seq_len, endpoint=False)
    return A * np.exp(-gamma * t) * np.cos(omega * t + phi)


# Sanity check
sample, _grange = generate_oscillators(5, seed=1991)
assert sample.shape == (5, SEQ_LEN)
assert sample.min() >= 0 and sample.max() < N_BINS
print(f'Oscillator generator: shape={sample.shape}, range=[{sample.min()}, {sample.max()}]')
print(f'Global range: ({_grange[0]:.3f}, {_grange[1]:.3f})')
del sample
print('Dataset generator ready (global discretization)')


In [ ]:
# Model Definitions -- SmallBERT and SmallMamba
# (Reused from Architectural_Control_Experiment.ipynb)

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class SmallBERT(nn.Module):
    """4-layer Transformer Encoder for Masked Value Prediction."""

    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN,
                 dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x, return_hidden=False):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        h = self.encoder(h)
        h = self.norm(h)
        logits = self.head(h)
        if return_hidden:
            return logits, h
        return logits


if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock

    class SmallMamba(nn.Module):
        """4-layer Mamba for Masked Value Prediction (native CUDA)."""

        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)
            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(MambaBlock(d_model=d_model, d_state=d_state,
                                              d_conv=d_conv, expand=expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)
            self._init_weights()

        def _init_weights(self):
            for name, p in self.named_parameters():
                if 'tok_emb' in name or 'pos_emb' in name:
                    nn.init.normal_(p, std=0.02)
                elif p.dim() > 1 and 'head' in name:
                    nn.init.xavier_uniform_(p)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            if return_hidden:
                return logits, h
            return logits

else:
    class SimpleSSMLayer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            d_inner = d_model * expand
            self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
            self.conv1d   = nn.Conv1d(d_inner, d_inner, d_conv,
                                      padding=d_conv - 1, groups=d_inner)
            self.D        = nn.Parameter(torch.ones(d_inner))
            self.out_proj = nn.Linear(d_inner, d_model, bias=False)

        def forward(self, x):
            B, L, D = x.shape
            xz = self.in_proj(x)
            x_inner, z = xz.chunk(2, dim=-1)
            x_conv = self.conv1d(x_inner.transpose(1, 2))[:, :, :L].transpose(1, 2)
            x_conv = torch.silu(x_conv)
            y = x_conv * torch.silu(z)
            y = y * self.D.unsqueeze(0).unsqueeze(0)
            return self.out_proj(y)

    class SmallMamba(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL,
                     num_layers=N_LAYERS, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop    = nn.Dropout(dropout)
            self.layers = nn.ModuleList()
            self.norms  = nn.ModuleList()
            for _ in range(num_layers):
                self.layers.append(SimpleSSMLayer(d_model, d_state, d_conv, expand))
                self.norms.append(nn.LayerNorm(d_model))
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)

        def forward(self, x, return_hidden=False):
            B, L = x.shape
            pos = torch.arange(L, device=x.device).unsqueeze(0)
            h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
            for layer, norm in zip(self.layers, self.norms):
                h = h + layer(norm(h))
            h = self.final_norm(h)
            logits = self.head(h)
            if return_hidden:
                return logits, h
            return logits

    print('NOTE: Using pure-PyTorch SSM fallback')



# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        # No weight tying (vocab_size=258 != d_model=256 causes shape mismatch)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


# =====================================================================
# Model registry (FIX: moved after SmallStripedHyena definition)
# =====================================================================
MODEL_CLASSES = {'SmallBERT': SmallBERT, 'SmallMamba': SmallMamba, 'SmallStripedHyena': SmallStripedHyena}
for name, cls in MODEL_CLASSES.items():
    m = cls()
    print(f'{name:12s}: {sum(p.numel() for p in m.parameters())/1e6:.2f}M params')
    del m
print('Model definitions ready')


In [ ]:
# Training Loop -- Causal Language Modeling (CLM)
# Switched from Masked Value Prediction to CLM for consistency
# with Architectural Control and Coupled Lorenz experiments.

import time
import gc
from torch.utils.data import DataLoader, TensorDataset


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def create_causal_batch(x):
    '''CLM-style shifted input/target pairs.'''
    return x[:, :-1], x[:, 1:]


def train_model(model, train_data, val_data, arch_name, dataset_name,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                weight_decay=WEIGHT_DECAY):
    model = model.to(DEVICE)

    train_ds = TensorDataset(torch.from_numpy(train_data).long())
    val_ds   = TensorDataset(torch.from_numpy(val_data).long())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              drop_last=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                              num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs * len(train_loader)
    )
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    best_state = None
    train_losses, val_losses = [], []

    ckpt_path = f'{CKPT_DIR}/{arch_name}_{dataset_name}_clm_best.pt'

    if os.path.exists(ckpt_path):
        print(f'  Loading existing checkpoint: {ckpt_path}')
        try:
            model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
            return model, [], []
        except RuntimeError as e:
            print(f'  Checkpoint shape mismatch, retraining: {e}')
            os.remove(ckpt_path)

    print(f'  Training {arch_name} on {dataset_name} ({epochs} epochs, CLM)...')
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss, n_batches = 0.0, 0

        for (batch_x,) in train_loader:
            batch_x = batch_x.to(DEVICE)
            inputs, targets = create_causal_batch(batch_x)
            logits = model(inputs)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            n_batches += 1

        avg_train = epoch_loss / n_batches
        train_losses.append(avg_train)

        model.eval()
        val_loss, val_batches = 0.0, 0
        with torch.no_grad():
            for (batch_x,) in val_loader:
                batch_x = batch_x.to(DEVICE)
                inputs, targets = create_causal_batch(batch_x)
                logits = model(inputs)
                loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))
                val_loss += loss.item()
                val_batches += 1

        avg_val = val_loss / max(val_batches, 1)
        val_losses.append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 5 == 0 or epoch == 0:
            elapsed = time.time() - start
            print(f'    Epoch {epoch+1:3d}/{epochs}  '
                  f'train={avg_train:.4f}  val={avg_val:.4f}  '
                  f'best_val={best_val_loss:.4f}  [{elapsed:.0f}s]')

    if best_state is not None:
        model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt_path)

    elapsed = time.time() - start
    print(f'  Done in {elapsed/60:.1f} min  (best val loss: {best_val_loss:.4f})')
    return model, train_losses, val_losses


@torch.no_grad()
def extract_embeddings(model, sequences, batch_size=128):
    '''Extract mean-pooled last-layer hidden states.'''
    model.eval()
    all_embs = []
    for i in range(0, len(sequences), batch_size):
        batch = torch.from_numpy(sequences[i:i+batch_size]).long().to(DEVICE)
        _, hidden = model(batch, return_hidden=True)
        emb = hidden.mean(dim=1)
        all_embs.append(emb.cpu().numpy())
    return np.concatenate(all_embs, axis=0)


print('Training loop and embedding extraction ready (CLM)')


In [ ]:
# Train All Models on Oscillator Data

print('Generating training data...')
train_data, OSCILLATOR_RANGE = generate_oscillators(N_TRAIN, seed=SEED)
eval_data, _ = generate_oscillators(N_EVAL, seed=SEED + 1991, global_range=OSCILLATOR_RANGE)
print(f'  Train: {train_data.shape}, Eval: {eval_data.shape}')
print(f'  Global range: ({OSCILLATOR_RANGE[0]:.3f}, {OSCILLATOR_RANGE[1]:.3f})')

trained_models = {}

for arch_name in ARCHITECTURES:
    print(f'\n{"="*60}')
    print(f'{arch_name}')
    print('=' * 60)

    model = MODEL_CLASSES[arch_name]()
    model, t_losses, v_losses = train_model(
        model, train_data, eval_data, arch_name, 'oscillator_interp'
    )
    trained_models[arch_name] = model
    cleanup_gpu()

print('\nAll models trained and ready for interpolation')


In [ ]:
# Interpolation Path Generator
#
# Both signals A and B are discretized using the SAME global range
# (from training data). This ensures linear interpolation in token space
# corresponds to linear interpolation in physical space.

def generate_interpolation_pair(seed_offset=0, global_range=None):
    '''Generate two distinct oscillator trajectories with different parameters.

    Both signals are discretized using the same global_range so that
    the same physical amplitude maps to the same bin in both sequences.
    '''
    rng = np.random.default_rng(SEED + seed_offset)
    t = np.linspace(0, 4, SEQ_LEN, endpoint=False)

    # Trajectory A: low frequency, light damping
    A_amp   = rng.uniform(0.5, 2.0)
    A_gamma = rng.uniform(0.2, 0.8)
    A_omega = rng.uniform(2.0, 8.0)
    A_phi   = rng.uniform(0, 2*np.pi)
    sig_A = A_amp * np.exp(-A_gamma * t) * np.cos(A_omega * t + A_phi)

    # Trajectory B: higher frequency, heavier damping
    B_amp   = rng.uniform(0.5, 2.0)
    B_gamma = rng.uniform(1.0, 2.0)
    B_omega = rng.uniform(10.0, 20.0)
    B_phi   = rng.uniform(0, 2*np.pi)
    sig_B = B_amp * np.exp(-B_gamma * t) * np.cos(B_omega * t + B_phi)

    # Use a shared range for both signals.
    # If a dataset-level range is provided, use that. Otherwise compute
    # a joint range from both signals so they're at least consistent
    # with each other.
    if global_range is not None:
        gmin, gmax = global_range
    else:
        gmin = min(sig_A.min(), sig_B.min())
        gmax = max(sig_A.max(), sig_B.max())

    seq_A = discretize(sig_A, global_min=gmin, global_max=gmax)
    seq_B = discretize(sig_B, global_min=gmin, global_max=gmax)

    params_A = dict(A=A_amp, gamma=A_gamma, omega=A_omega, phi=A_phi)
    params_B = dict(A=B_amp, gamma=B_gamma, omega=B_omega, phi=B_phi)

    return seq_A, seq_B, sig_A, sig_B, params_A, params_B


def linear_interpolation_path(seq_A, seq_B, n_steps=N_INTERP_STEPS):
    '''Create interpolation path from A to B in token space.

    Returns:
        path: np.ndarray (n_steps, seq_len) int64
        alphas: np.ndarray (n_steps,) float64
    '''
    alphas = np.linspace(0.0, 1.0, n_steps)
    path = np.zeros((n_steps, len(seq_A)), dtype=np.int64)

    for i, alpha in enumerate(alphas):
        interp = (1 - alpha) * seq_A.astype(np.float64) + alpha * seq_B.astype(np.float64)
        path[i] = np.clip(np.round(interp).astype(np.int64), 0, N_BINS - 1)

    return path, alphas


# Generate all interpolation pairs using the training data's global range
print(f'Generating {N_INTERP_PAIRS} interpolation pairs...')
print(f'Using global range from training data: ({OSCILLATOR_RANGE[0]:.3f}, {OSCILLATOR_RANGE[1]:.3f})')
interp_data = []
for pair_idx in range(N_INTERP_PAIRS):
    seq_A, seq_B, sig_A, sig_B, pA, pB = generate_interpolation_pair(
        seed_offset=pair_idx * 100, global_range=OSCILLATOR_RANGE)
    path, alphas = linear_interpolation_path(seq_A, seq_B)
    interp_data.append({
        'seq_A': seq_A, 'seq_B': seq_B,
        'sig_A': sig_A, 'sig_B': sig_B,
        'params_A': pA, 'params_B': pB,
        'path': path, 'alphas': alphas,
    })
    hamming = np.sum(seq_A != seq_B)
    print(f'  Pair {pair_idx}: Hamming distance = {hamming}/{SEQ_LEN} '
          f'({hamming/SEQ_LEN*100:.1f}%)')

print(f'\n{len(interp_data)} interpolation pairs ready')


In [ ]:
# Embed Interpolation Paths + Compute Lipschitz Profiles

from evaluation_harness import compute_lipschitz_profile
from scipy.spatial.distance import cosine


def embed_interpolation_path(model, path):
    """Embed each sequence in the interpolation path.

    Args:
        model: trained model with forward(x, return_hidden=True)
        path: np.ndarray (n_steps, seq_len) int64

    Returns:
        np.ndarray (n_steps, d_model) -- mean-pooled embeddings
    """
    return extract_embeddings(model, path, batch_size=N_INTERP_STEPS)


def compute_cosine_from_start(embeddings):
    """Cosine distance of each step from the starting embedding."""
    start = embeddings[0]
    dists = [cosine(start, emb) if np.isfinite(cosine(start, emb)) else 0.0
             for emb in embeddings]
    return np.array(dists)


# =====================================================================
# Embed all pairs with both models
# =====================================================================
results = {}  # arch -> {'embeddings': list, 'lipschitz': list, 'cosine': list}

for arch_name in ARCHITECTURES:
    print(f'\n{"="*60}')
    print(f'Embedding interpolation paths: {arch_name}')
    print('=' * 60)

    model = trained_models[arch_name]
    model.eval()

    all_embs = []
    all_lipschitz = []
    all_cosine = []

    for pair_idx, data in enumerate(interp_data):
        cache_path = f'{CACHE_DIR}/{arch_name}_interp_pair{pair_idx}.npy'

        if os.path.exists(cache_path):
            embs = np.load(cache_path)
        else:
            print(f'  Pair {pair_idx}...')
            embs = embed_interpolation_path(model, data['path'])
            np.save(cache_path, embs)

        lip = compute_lipschitz_profile(embs)
        cos_dist = compute_cosine_from_start(embs)

        all_embs.append(embs)
        all_lipschitz.append(lip)
        all_cosine.append(cos_dist)

    results[arch_name] = {
        'embeddings': all_embs,
        'lipschitz': all_lipschitz,
        'cosine': all_cosine,
    }

    # Compute aggregate stats
    mean_lip = np.mean([lip.mean() for lip in all_lipschitz])
    max_lip  = np.mean([lip.max() for lip in all_lipschitz])
    std_lip  = np.mean([lip.std() for lip in all_lipschitz])
    print(f'  Mean Lipschitz: {mean_lip:.4f}')
    print(f'  Max  Lipschitz: {max_lip:.4f}')
    print(f'  Std  Lipschitz: {std_lip:.4f}')

    # Save aggregate Lipschitz profile
    np.save(f'{CACHE_DIR}/{arch_name}_lipschitz_mean.npy',
            np.mean(np.array(all_lipschitz), axis=0))

print('\nAll interpolation paths embedded and Lipschitz profiles computed')

In [ ]:
# Export Lipschitz Profiles to CSV
# This CSV is used by the figure-generation script (figs/figure1_ground_truth.py)

import pandas as pd

csv_rows = []
for arch_name in ARCHITECTURES:
    all_lip = np.array(results[arch_name]['lipschitz'])  # (n_pairs, n_steps-1)
    mean_lip = all_lip.mean(axis=0)
    std_lip = all_lip.std(axis=0)
    alphas_mid = alphas[:-1] + 0.5 / (N_INTERP_STEPS - 1)  # midpoints
    for step_idx in range(len(mean_lip)):
        csv_rows.append({
            'architecture': arch_name,
            'step': step_idx,
            'alpha': alphas_mid[step_idx],
            'cosine_lipschitz_mean': mean_lip[step_idx],
            'cosine_lipschitz_std': std_lip[step_idx],
        })

df_lip = pd.DataFrame(csv_rows)
lip_csv_path = f'{RESULTS_DIR}/track_a_lipschitz.csv'
df_lip.to_csv(lip_csv_path, index=False)
print(f'Lipschitz CSV saved: {lip_csv_path}')
print(f'  {len(df_lip)} rows ({len(ARCHITECTURES)} archs x {len(df_lip)//len(ARCHITECTURES)} steps)')

# Summary stats
summary_rows = []
for arch_name in ARCHITECTURES:
    all_lip = np.array(results[arch_name]['lipschitz'])
    summary_rows.append({
        'architecture': arch_name,
        'mean_lipschitz': all_lip.mean(),
        'max_lipschitz': np.mean([l.max() for l in results[arch_name]['lipschitz']]),
        'std_lipschitz': all_lip.std(),
        'n_pairs': len(results[arch_name]['lipschitz']),
    })

df_summary = pd.DataFrame(summary_rows)
summary_path = f'{RESULTS_DIR}/track_a_summary.csv'
df_summary.to_csv(summary_path, index=False)
print(f'Summary CSV saved: {summary_path}')
print(df_summary.to_string(index=False))


In [ ]:
# Visualization -- 2 rows (BERT, Mamba) x 4 columns

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

plt.rcParams.update({'font.size': 11})

ARCH_COLORS = {'SmallBERT': '#2563EB', 'SmallMamba': '#16A34A', 'SmallStripedHyena': '#7C3AED'}

fig, axes = plt.subplots(len(ARCHITECTURES), 4, figsize=(22, 5 * len(ARCHITECTURES)))

# Use first interpolation pair for panels A-C, aggregate for panel D
pair_idx = 0
data0 = interp_data[pair_idx]
alphas = data0['alphas']
t = np.linspace(0, 4, SEQ_LEN, endpoint=False)

for row, arch_name in enumerate(ARCHITECTURES):
    color = ARCH_COLORS[arch_name]
    embs = results[arch_name]['embeddings'][pair_idx]
    lip  = results[arch_name]['lipschitz'][pair_idx]
    cos_d = results[arch_name]['cosine'][pair_idx]

    # --- Panel A: Input signals (raw waveforms) ---
    ax = axes[row, 0]
    ax.plot(t, data0['sig_A'], color='#2563EB', alpha=0.8, label='Seq A', linewidth=1.5)
    ax.plot(t, data0['sig_B'], color='#DC2626', alpha=0.8, label='Seq B', linewidth=1.5)
    # Show 3 midpoints
    for mid_alpha in [0.25, 0.5, 0.75]:
        mid_idx = int(mid_alpha * (N_INTERP_STEPS - 1))
        mid_seq = data0['path'][mid_idx]
        # Reconstruct approximate continuous signal from bins
        mid_cont = mid_seq / (N_BINS - 1) * (data0['sig_A'].max() - data0['sig_A'].min()) + data0['sig_A'].min()
        ax.plot(t, mid_cont, color='gray', alpha=0.4, linewidth=0.8)
    ax.set_xlabel('Time')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'{arch_name} -- Input Signals', fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.2)

    # --- Panel B: PCA of embedding trajectory ---
    ax = axes[row, 1]
    pca = PCA(n_components=2)
    embs_2d = pca.fit_transform(embs)
    scatter = ax.scatter(embs_2d[:, 0], embs_2d[:, 1], c=alphas, cmap='coolwarm',
                         s=25, alpha=0.8, edgecolors='white', linewidth=0.3)
    ax.plot(embs_2d[:, 0], embs_2d[:, 1], color='gray', alpha=0.3, linewidth=0.8)
    # Mark start and end
    ax.scatter(embs_2d[0, 0], embs_2d[0, 1], c='blue', s=100, marker='s',
               zorder=10, edgecolors='black', linewidth=1.5, label='A (start)')
    ax.scatter(embs_2d[-1, 0], embs_2d[-1, 1], c='red', s=100, marker='D',
               zorder=10, edgecolors='black', linewidth=1.5, label='B (end)')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{arch_name} -- Embedding PCA', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    plt.colorbar(scatter, ax=ax, label='alpha')

    # --- Panel C: Cosine distance from start ---
    ax = axes[row, 2]
    ax.plot(alphas, cos_d, color=color, linewidth=2)
    ax.fill_between(alphas, cos_d, alpha=0.15, color=color)
    ax.set_xlabel('Interpolation alpha')
    ax.set_ylabel('Cosine Distance from A')
    ax.set_title(f'{arch_name} -- Cosine Distance', fontweight='bold')
    ax.grid(alpha=0.2)

    # --- Panel D: Lipschitz profile (averaged over all pairs) ---
    ax = axes[row, 3]
    all_lip = np.array(results[arch_name]['lipschitz'])
    mean_lip = all_lip.mean(axis=0)
    std_lip  = all_lip.std(axis=0)
    lip_alphas = alphas[:-1] + 0.5 / (N_INTERP_STEPS - 1)  # midpoints

    ax.plot(lip_alphas, mean_lip, color=color, linewidth=2,
            label=f'mean (n={N_INTERP_PAIRS})')
    ax.fill_between(lip_alphas, mean_lip - std_lip, mean_lip + std_lip,
                    alpha=0.2, color=color)
    ax.set_xlabel('Interpolation alpha')
    ax.set_ylabel('Local Lipschitz Constant')
    ax.set_title(f'{arch_name} -- Lipschitz Profile', fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.suptitle('Track A: Interpolation Smoothness -- SmallBERT vs SmallStripedHyena vs SmallMamba\n'
             'Damped Harmonic Oscillator',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/track_a_interpolation.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')
plt.show()

# --- Summary statistics ---
print('\n' + '=' * 60)
print('INTERPOLATION SMOOTHNESS SUMMARY')
print('=' * 60)
for arch_name in ARCHITECTURES:
    all_lip = np.array(results[arch_name]['lipschitz'])
    print(f'\n{arch_name}:')
    print(f'  Mean Lipschitz:     {all_lip.mean():.4f}')
    print(f'  Max  Lipschitz:     {np.mean([l.max() for l in results[arch_name]["lipschitz"]]):.4f}')
    print(f'  Std  Lipschitz:     {all_lip.std():.4f}')
    print(f'  Smoothness ratio:   {all_lip.mean() / (np.mean([l.max() for l in results[arch_name]["lipschitz"]]) + 1e-8):.4f}')

# Pairwise Lipschitz ratios
arch_means = {a: np.array(results[a]['lipschitz']).mean() for a in ARCHITECTURES}
print('\nPairwise Lipschitz ratios:')
for i, a1 in enumerate(ARCHITECTURES):
    for a2 in ARCHITECTURES[i+1:]:
        ratio = arch_means[a2] / (arch_means[a1] + 1e-8)
        print(f'  {a2}/{a1}: {ratio:.4f}')
smoothest = min(arch_means, key=arch_means.get)
roughest = max(arch_means, key=arch_means.get)
print(f'\nSmoothest: {smoothest} (mean Lipschitz={arch_means[smoothest]:.4f})')
print(f'Roughest:  {roughest} (mean Lipschitz={arch_means[roughest]:.4f})')

In [ ]:
# Phase Space Overlay -- Direct Comparison of All Architecture Paths
#
# Project ALL architectures into the SAME PCA space to visually compare
# the smoothness of their interpolation paths.

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

pair_idx = 0
ARCH_COLORS_OVERLAY = {'SmallBERT': '#DC2626', 'SmallMamba': '#16A34A', 'SmallStripedHyena': '#7C3AED'}
ARCH_CMAPS = {'SmallBERT': 'Reds', 'SmallMamba': 'Greens', 'SmallStripedHyena': 'Purples'}
ARCH_LABELS = {'SmallBERT': 'BERT (Transformer)', 'SmallMamba': 'Mamba (SSM)', 'SmallStripedHyena': 'StripedHyena (Hybrid)'}

# Gather embeddings and Lipschitz for all architectures
arch_embs = {a: results[a]['embeddings'][pair_idx] for a in ARCHITECTURES}
arch_lips = {a: results[a]['lipschitz'][pair_idx] for a in ARCHITECTURES}

# --- Panel A: Phase Space Overlay (Joint PCA) ---
ax = axes[0]

# Fit PCA on concatenated embeddings so all use same PC basis
all_embs_concat = np.vstack([arch_embs[a] for a in ARCHITECTURES])
pca = PCA(n_components=2)
all_2d = pca.fit_transform(all_embs_concat)

# Split back
offset = 0
arch_2d = {}
for a in ARCHITECTURES:
    n = len(arch_embs[a])
    arch_2d[a] = all_2d[offset:offset+n]
    offset += n

for a in ARCHITECTURES:
    pts = arch_2d[a]
    color = ARCH_COLORS_OVERLAY[a]
    ax.plot(pts[:, 0], pts[:, 1], color=color, linewidth=2.5,
            alpha=0.8, label=ARCH_LABELS[a], zorder=3)
    ax.scatter(pts[:, 0], pts[:, 1], c=alphas, cmap=ARCH_CMAPS[a],
               s=20, alpha=0.5, edgecolors=color, linewidth=0.3, zorder=4)

# Mark start/end from first architecture
first_pts = arch_2d[ARCHITECTURES[0]]
ax.scatter(first_pts[0, 0], first_pts[0, 1], c='blue', s=200, marker='s',
           zorder=10, edgecolors='black', linewidth=2, label='Start (A)')
ax.scatter(first_pts[-1, 0], first_pts[-1, 1], c='red', s=200, marker='D',
           zorder=10, edgecolors='black', linewidth=2, label='End (B)')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax.set_title('A. Phase Space: Geometric Roughness', fontweight='bold', fontsize=13)
ax.legend(fontsize=8, loc='best')
ax.grid(alpha=0.15)

# --- Panel B: Lipschitz Comparison (Direct Overlay) ---
ax = axes[1]
lip_alphas = alphas[:-1] + 0.5 / (N_INTERP_STEPS - 1)

for a in ARCHITECTURES:
    all_lip = np.array(results[a]['lipschitz'])
    mean_lip = all_lip.mean(axis=0)
    std_lip = all_lip.std(axis=0)
    color = ARCH_COLORS_OVERLAY[a]
    ax.plot(lip_alphas, mean_lip, color=color, linewidth=2.5,
            label=f'{a} (mean={mean_lip.mean():.3f})', zorder=3)
    ax.fill_between(lip_alphas, mean_lip - std_lip, mean_lip + std_lip,
                    color=color, alpha=0.12, zorder=2)

ax.set_xlabel('Interpolation alpha', fontsize=12)
ax.set_ylabel('Local Lipschitz Constant', fontsize=12)
ax.set_title('B. Lipschitz Profile Comparison', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.15)

# --- Panel C: Lipschitz Distribution (Histogram) ---
ax = axes[2]

for a in ARCHITECTURES:
    lip_flat = np.array(results[a]['lipschitz']).flatten()
    color = ARCH_COLORS_OVERLAY[a]
    ax.hist(lip_flat, bins=40, color=color, alpha=0.4, label=a,
            density=True, edgecolor=color, linewidth=0.8)
    ax.axvline(lip_flat.mean(), color=color, linestyle='--', linewidth=2,
               label=f'{a} mean: {lip_flat.mean():.3f}')

ax.set_xlabel('Local Lipschitz Constant', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('C. Lipschitz Distribution (All Pairs)', fontweight='bold', fontsize=13)
ax.legend(fontsize=8)
ax.grid(alpha=0.15, axis='y')

fig.suptitle('Track A: Phase Space Analysis -- Architecture Comparison\n'
             'Geometric Roughness of Interpolation Paths',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()

fig_path = f'{RESULTS_DIR}/track_a_phase_space_overlay.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {fig_path}')
plt.show()

print('\n' + '=' * 70)
print('PHASE SPACE VERDICT')
print('=' * 70)
arch_lip_means = {}
for a in ARCHITECTURES:
    lip_flat = np.array(results[a]['lipschitz']).flatten()
    arch_lip_means[a] = lip_flat.mean()
    print(f'  {a:25s}: mean Lipschitz = {lip_flat.mean():.4f}, std = {lip_flat.std():.4f}')

smoothest = min(arch_lip_means, key=arch_lip_means.get)
roughest = max(arch_lip_means, key=arch_lip_means.get)
ratio = arch_lip_means[roughest] / (arch_lip_means[smoothest] + 1e-8)
print(f'\n  Smoothest: {smoothest}')
print(f'  Roughest:  {roughest} ({ratio:.2f}x higher mean Lipschitz)')
print('=' * 70)
